# 02a_lazy_join: Join multiple tables + inspect the plan

Goal: use Polars `LazyFrame` joins to connect the wearable tables and compute a *sleep ↔ physiology* summary.

Data relationships:

- `user_profile.user_id` joins to `sleep_diary.user_id`
- `sensor_hrv.device_id` encodes the user id suffix (e.g., `DEV-00012` → `USER-00012`)

We’ll build a nightly physiology table first (aggregate the large table), then join to the smaller tables. This keeps the join grain under control.

## Data setup

We’ll create `data/` and `outputs/`, then generate the synthetic dataset if it’s missing.

In [1]:
from pathlib import Path
import subprocess
import sys

Path("data").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

sensor_dir = Path("data/sensor_hrv")
if not sensor_dir.exists() or not list(sensor_dir.glob("*.parquet")):
    subprocess.run(
        [sys.executable, "generate_demo_data.py", "--size", "small", "--output-dir", "data"],
        check=True,
    )

## Steps

In [2]:
from pathlib import Path

sensor_dir = Path("data/sensor_hrv")
required_files = [
    Path("data/sleep_diary.parquet"),
    Path("data/user_profile.parquet"),
]

missing_files = [p for p in required_files if not p.exists()]
sensor_parts = list(sensor_dir.glob("*.parquet"))

if missing_files or not sensor_parts:
    raise FileNotFoundError(
        "Missing demo data.\n"
        f"- Missing files: {', '.join(str(p) for p in missing_files) if missing_files else '(none)'}\n"
        f"- Sensor parts found: {len(sensor_parts)}\n"
        "Run: uv run python generate_demo_data.py --size small --output-dir data"
    )

### 1) Create LazyFrames

In [3]:
import polars as pl

users = pl.scan_parquet("data/user_profile.parquet")
sleep = pl.scan_parquet("data/sleep_diary.parquet")
sensor = pl.scan_parquet("data/sensor_hrv/*.parquet")

# Small previews (these stay lazy until collected)
users.head().collect()
sleep.head().collect()

user_id,date,bedtime,fall_asleep_time,wake_time,waso_minutes,sleep_duration,sleep_efficiency,sleep_latency
str,date,time,time,time,i64,f64,f64,f64
"""USER-00000""",2024-01-01,20:00:00,20:08:00,02:43:50.196773,0,6.6,98.0,8.0
"""USER-00000""",2024-01-02,21:00:00,21:00:00,02:10:00.675403,0,5.17,100.0,0.0
"""USER-00000""",2024-01-03,23:00:00,23:00:00,07:31:18.795751,0,8.52,100.0,0.0
"""USER-00000""",2024-01-04,20:00:00,20:00:00,02:17:45.045423,0,6.3,100.0,0.0
"""USER-00000""",2024-01-05,23:00:00,23:00:00,04:41:53.358124,0,5.7,100.0,0.0


### 2) Normalize keys (derive `user_id` from `device_id`)

In [4]:
import polars as pl

sensor_keyed = sensor.with_columns(
    pl.concat_str([pl.lit("USER-"), pl.col("device_id").str.split("-").list.get(1)]).alias("user_id"),
    pl.col("ts_start").dt.date().alias("date"),
)

### 3) Join sleep diary to nightly physiology

We join on `user_id` + `date`, then summarize by occupation.

In [5]:
import polars as pl

night_segments = (
    sensor_keyed
    .filter(pl.col("missingness_score") <= 0.35)
    .filter(pl.col("ts_start").dt.hour().is_between(22, 23) | pl.col("ts_start").dt.hour().is_between(0, 6))
    .select(["user_id", "date", "hrv_sdnn", "hrv_rmssd", "heart_rate", "steps"])
)

nightly = (
    night_segments
    .group_by(["user_id", "date"])
    .agg([
        pl.len().alias("num_segments"),
        pl.mean("heart_rate").alias("night_mean_hr"),
        pl.mean("hrv_sdnn").alias("night_mean_sdnn"),
        pl.mean("hrv_rmssd").alias("night_mean_rmssd"),
        pl.sum("steps").alias("night_steps"),
    ])
)

joined = (
    sleep
    .join(nightly, on=["user_id", "date"], how="inner")
    .join(users.select(["user_id", "age", "gender", "occupation"]), on="user_id", how="left")
)

summary = (
    joined
    .group_by(["occupation", "gender"])
    .agg([
        pl.len().alias("n_nights"),
        pl.mean("sleep_efficiency").alias("avg_sleep_efficiency"),
        pl.mean("night_mean_sdnn").alias("avg_night_sdnn"),
        pl.corr("sleep_efficiency", "night_mean_sdnn").alias("corr_sleep_sdnn"),
    ])
    .sort(["occupation", "gender"])
)

print(summary.explain())
out = summary.collect(engine="streaming")
out.write_parquet("outputs/sleep_hrv_by_group.parquet")
out

# A small inline table helps in the notebook
out.head(10)

SORT BY [col("occupation"), col("gender")]
  AGGREGATE[maintain_order: false]
    [len().alias("n_nights"), col("sleep_efficiency").mean().alias("avg_sleep_efficiency"), col("night_mean_sdnn").mean().alias("avg_night_sdnn"), col("sleep_efficiency").pearson_correlation([col("night_mean_sdnn")]).alias("corr_sleep_sdnn")] BY [col("occupation"), col("gender")]
    FROM
    simple π 4/4 ["sleep_efficiency", ... 3 other columns]
      LEFT JOIN:
      LEFT PLAN ON: [col("user_id")]
        simple π 3/3 ["user_id", "sleep_efficiency", ... 1 other column]
          INNER JOIN:
          LEFT PLAN ON: [col("user_id"), col("date")]
            Parquet SCAN [data/sleep_diary.parquet]
            PROJECT 3/9 COLUMNS
            ESTIMATED ROWS: 5250
          RIGHT PLAN ON: [col("user_id"), col("date")]
            AGGREGATE[maintain_order: false]
              [len().alias("num_segments"), col("heart_rate").mean().alias("night_mean_hr"), col("hrv_sdnn").mean().alias("night_mean_sdnn"), col("hrv_rm

occupation,gender,n_nights,avg_sleep_efficiency,avg_night_sdnn,corr_sleep_sdnn
str,str,u32,f64,f64,f64
"""Graduate""","""Female""",455,97.778242,88.480233,0.018264
"""Graduate""","""Male""",350,97.618857,84.755865,0.048684
"""Graduate""","""Nonbinary""",420,97.703095,90.409849,0.134085
"""Office Worker""","""Female""",455,97.801538,87.21482,-0.005882
"""Office Worker""","""Male""",455,97.829451,87.959787,0.084699
"""Office Worker""","""Nonbinary""",350,98.094857,90.489257,-0.023013
"""Other""","""Female""",350,97.988,89.630647,0.055476
"""Other""","""Male""",665,98.019699,87.809033,0.00378
"""Other""","""Nonbinary""",490,98.095714,86.273009,0.04198


## Visual: sleep efficiency vs nightly HRV (sample)

We’ll plot a sample of joined nights to see the shape of the relationship.

In [6]:
import altair as alt

sample = joined.select([
    "sleep_efficiency",
    "night_mean_sdnn",
    "occupation",
    "gender",
]).collect(engine="streaming").sample(n=2000, shuffle=True)

alt.Chart(sample.to_pandas()).mark_circle(opacity=0.25).encode(
    x=alt.X("sleep_efficiency:Q", title="Sleep efficiency"),
    y=alt.Y("night_mean_sdnn:Q", title="Night mean SDNN"),
    color="gender:N",
    tooltip=["occupation", "gender"],
).properties(width=650, height=300)

alt.Chart(...)

## Checkpoints

- `outputs/sleep_hrv_by_group.parquet` exists
- `summary.explain()` shows projection/predicate pushdown before joins
- Deriving `user_id` from `device_id` makes the join possible